# EXP-0002: Static Pre-Race Starting Grid Benchmark

**Project:** APEX — F1 Intelligence  
**Workflow:** Collaborative Research Workspace (Project Owner + ChatGPT)  
**Experiment ID:** `EXP-0002`  
**Creation Date:** `2026-10-19`  
**Target Research Question(s):** `RQ1` ([docs/research/RESEARCH_QUESTIONS.md](../docs/research/RESEARCH_QUESTIONS.md))  
**Target Hypothesis ID(s):**  
- [HYP-0001](../docs/research/HYPOTHESES.md#hyp-0001-sequential-information-superiority): Sequential Information Superiority  
- [HYP-GRID-001](../docs/research/HYPOTHESES.md#hyp-grid-001-pre-race-grid-position-dominance): Pre-Race Grid Position Dominance  
- [HYP-GRID-002](../docs/research/HYPOTHESES.md#hyp-grid-002-nonlinear-grid-representation-superiority): Nonlinear Grid Representation Superiority  
**Forecast Checkpoint:** $T_5$ (Pre-Race, ~30 minutes prior to formation lap / official FIA Final Starting Grid)  
**Status:** `SCAFFOLDED` (Candidate audit complete; notebook ready for collaborative execution)

## 1. Scientific Motivation & Tested Claims

In Formula 1 forecasting, the starting grid is historically the single most powerful static predictor of race victory. However, sports forecasting models frequently commit subtle methodology flaws: leaking post-race DNS information into pre-race entrant pools, improperly treating pit-lane starters (`grid == 0`), or failing to enforce strictly proper scoring rules across the entire candidate simplex $\sum P = 1$.

### Research Objectives:
1. Establish the **empirical grid-position-only baseline** ($P(\text{Win} \mid \text{Grid})$) using historical transition frequencies with Dirichlet/Laplace smoothing.
2. Evaluate a **multinomial logit softmax model** parameterized with nonlinear grid transformations ($\log(\text{grid})$, front-row indicator, team prior).
3. Prototype a **tree-based GBDT benchmark (LightGBM)** to test whether nonlinear interactions and championship context reduce log loss beyond empirical grid tables.
4. Compare all models against the static pre-weekend baseline (**EXP-0001-R3 Elo**, Log Loss: 2.278177, Brier: 0.845726) using strictly proper scoring rules on identical out-of-sample test cohorts.

## 2. Pre-Race Candidate Field Protocol & Anti-Leakage Guardrails

Per the empirical audit documented in [docs/research/CANDIDATE_FIELD_AUDIT.md](../docs/research/CANDIDATE_FIELD_AUDIT.md), all models in this notebook adhere to strict temporal integrity and candidate-definition protocols:

> **Provisional Result-Record Proxy Caveat:** The current dataset relies on retrospective Ergast/Jolpica race-result records. We have not established an independently timestamped pre-race entry list for every race. This field is a **provisional result-record proxy** until entry-field reconstruction is verified against official pre-session timing manifests. Zero post-race leakage cannot be formally claimed as proven. Drivers must never be excluded using post-race status codes.

1. **Outcome-Independent Candidate Ordering:** Candidates within each race must be ordered deterministically by `driver_id` (or ascending `grid_position` with `driver_id` tie-break). Candidates must **never** be ordered by finishing position, status, or points.
2. **Race-Specific Grid-Zero Representation:** In Ergast/Jolpica, `grid == 0` is an API indicator. The empirical audit identified 49 of 56 zero-grid records as active pit-lane starts, while the remaining 7 require appropriate contextual interpretation or verification (and must not be treated as definitive pit-lane starts). These drivers remain in the candidate set, assigned a provisional race-specific effective grid position equal to the **maximum positive starting grid position in that race plus one** ($\text{effective\_grid} = \max_{i, \text{grid}_i > 0}(\text{grid}_i) + 1$), accompanied by an explicit `is_grid_zero` indicator.
3. **No Retrospective DNS Dropping:** Entrants who break down on the reconnaissance or formation lap held valid starting slots at $T_5$. Dropping them using post-race status codes (`"Hydraulics"`, `"Did not start"`) is direct post-cutoff outcome leakage. They must receive their pre-race probability share in the simplex $\sum P = 1$.

## 3. Environment Setup & Project Root Discovery

In [1]:
import os
import sys
from pathlib import Path

# Robust project root discovery searching for marker files
def find_project_root(markers=("AGENTS.md", "README.md")) -> Path:
    current = Path.cwd().resolve()
    for parent in [current] + list(current.parents):
        if all((parent / m).exists() for m in markers):
            return parent
    return current

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root resolved: {PROJECT_ROOT}")
assert (PROJECT_ROOT / "AGENTS.md").exists(), "AGENTS.md not found in project root!"

Project root resolved: /Users/dhruv/dhruv files/apex-f1


In [2]:
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from scipy.special import softmax

# Set global random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Standard data science stack loaded successfully.")

Standard data science stack loaded successfully.


## 4. Dataset Ingestion & Pre-Race Candidate Validation

We load `data/interim/driver_race_results_2010_2026_v2_regulation_metadata.csv` (7,285 rows across 346 races) and construct the candidate field at checkpoint $T_5$. Note that this represents a provisional result-record proxy until independently timestamped entry lists are verified.

In [3]:
DATA_PATH = PROJECT_ROOT / "data" / "interim" / "driver_race_results_2010_2026_v2_regulation_metadata.csv"
assert DATA_PATH.exists(), f"Source dataset not found at {DATA_PATH}"

df_raw = pd.read_csv(DATA_PATH)
df_raw["season_int"] = df_raw["season"].astype(int)
df_raw["round_int"] = df_raw["round"].astype(int)
df_raw["finish_position_int"] = df_raw["finish_position"].astype(int)
df_raw["grid_position_int"] = df_raw["grid_position"].astype(int)
df_raw["race_id"] = df_raw["season_int"].astype(str) + "_" + df_raw["round_int"].astype(str).str.zfill(2)

n_races = df_raw["race_id"].nunique()
min_s = df_raw["season_int"].min()
max_s = df_raw["season_int"].max()
print(f"Loaded raw dataset: {len(df_raw)} rows, {n_races} unique Grand Prix events.")
print(f"Seasons covered: {min_s} to {max_s}.")

Loaded raw dataset: 7285 rows, 346 unique Grand Prix events.
Seasons covered: 2010 to 2026.


### 4.1 Construct Pre-Race Starting Grid Features & Handle Grid-Zero Records

Grid zero is an API value whose underlying circumstances (identified as 49 pit-lane starts and 7 requiring further verification) must not be treated as definitive proof of a pit-lane start. We assign a provisional race-specific effective grid position equal to the maximum positive starting grid in that race plus one ($\max_{i, \text{grid}_i > 0}(\text{grid}_i) + 1$) and retain an explicit `is_grid_zero` indicator. Derived features (`log_grid`, `is_front_row`, `is_pole`) are recomputed consistently from `effective_grid`.

In [4]:
# Sort candidates strictly by outcome-independent identifiers: season, round, driver_id
df_candidates = df_raw.sort_values(by=["season_int", "round_int", "driver_id"]).reset_index(drop=True).copy()

# Standardize winner label (binary outcome y in {0, 1})
df_candidates["y"] = (df_candidates["finish_position_int"] == 1).astype(int)

# Encode grid-zero records and race-specific effective grid position
# Grid zero is an API value (49 of 56 audited as pit-lane starts; 7 require verification).
# Assign effective_grid as the maximum positive starting grid in that race + 1.
max_pos_grid_per_race = (
    df_candidates[df_candidates["grid_position_int"] > 0]
    .groupby("race_id")["grid_position_int"]
    .max()
)
race_max_pos = df_candidates["race_id"].map(max_pos_grid_per_race)

df_candidates["is_grid_zero"] = (df_candidates["grid_position_int"] == 0).astype(int)
df_candidates["effective_grid"] = np.where(
    df_candidates["grid_position_int"] > 0,
    df_candidates["grid_position_int"],
    race_max_pos + 1
).astype(float)

# Feature engineering: nonlinear grid transformations recomputed consistently from effective_grid
df_candidates["log_grid"] = np.log(df_candidates["effective_grid"].astype(float))
df_candidates["is_front_row"] = (df_candidates["effective_grid"].isin([1.0, 2.0])).astype(int)
df_candidates["is_pole"] = (df_candidates["effective_grid"] == 1.0).astype(int)

# Verify exactly one winner per Grand Prix
winners_per_race = df_candidates.groupby("race_id")["y"].sum()
assert (winners_per_race == 1).all(), f"Discrepancy in winner counts: {(winners_per_race != 1).sum()} races with != 1 winner!"

n_grid_zero = int(df_candidates["is_grid_zero"].sum())
print(f"Pre-race candidate field (provisional result-record proxy): {len(df_candidates)} entrants across {len(winners_per_race)} races.")
print(f"Total API grid-zero entrants encoded: {n_grid_zero} (effective grid: race max positive + 1; is_grid_zero flag set).")

Pre-race candidate field (provisional result-record proxy): 7285 entrants across 346 races.
Total API grid-zero entrants encoded: 56 (effective grid: race max positive + 1; is_grid_zero flag set).


## 5. Evaluation Strategy: Scoped Cohorts & Chronological Splits

Per `CANDIDATE_FIELD_AUDIT.md`:
- **Primary Benchmark Cohort:** 2018–2024 (149 rounds, modern 20-car grid, high regulatory consistency).
- **Historical Warm-up / Sensitivity Cohort:** 2010–2017 (156 rounds, V8/V6 transition).
- **Walk-Forward Evaluation:** Sequential rolling-origin training on past completed seasons.

In [5]:
# Define cohort masks
PRIMARY_COHORT = (df_candidates["season_int"] >= 2018) & (df_candidates["season_int"] <= 2024)
WARMUP_COHORT = (df_candidates["season_int"] >= 2010) & (df_candidates["season_int"] <= 2017)

n_primary = df_candidates[PRIMARY_COHORT]["race_id"].nunique()
n_warmup = df_candidates[WARMUP_COHORT]["race_id"].nunique()
print(f"Primary Benchmark Cohort (2018-2024): {n_primary} rounds, {int(PRIMARY_COHORT.sum())} candidate rows.")
print(f"Warm-up Cohort (2010-2017): {n_warmup} rounds, {int(WARMUP_COHORT.sum())} candidate rows.")

Primary Benchmark Cohort (2018-2024): 149 rounds, 2979 candidate rows.
Warm-up Cohort (2010-2017): 156 rounds, 3453 candidate rows.


## 6. Strictly Proper Scoring Metrics

We define vector-level scoring metrics matching `EXP-0001` standards:
- Multi-class Log Loss: $-\frac{1}{R} \sum_{r=1}^R \log(p_{r, \text{winner}})$
- Multi-class Brier Score: $\frac{1}{R} \sum_{r=1}^R \sum_{i=1}^{N_r} (p_{r, i} - y_{r, i})^2$
- Tie-aware Top-1 Accuracy & Fractional Credit

In [6]:
def evaluate_race_predictions(race_preds_df: pd.DataFrame, prob_col: str = "win_prob") -> dict:
    """
    Evaluates multiclass predictions across races with strict simplex and tie handling.
    """
    log_losses = []
    brier_scores = []
    tie_aware_top1 = 0
    fractional_top1 = 0.0
    legacy_argmax_top1 = 0
    total_races = 0

    for race_id, group in race_preds_df.groupby("race_id", sort=False):
        p = group[prob_col].to_numpy(dtype=float)
        y = group["y"].to_numpy(dtype=int)
        
        # Assert valid probability distribution
        assert np.isclose(p.sum(), 1.0, atol=1e-4), f"Simplex violation in race {race_id}: sum = {p.sum()}"
        
        # Multiclass log loss for the true winner
        winner_idx = np.where(y == 1)[0]
        assert len(winner_idx) == 1
        p_winner = np.clip(p[winner_idx[0]], 1e-15, 1.0)
        log_losses.append(-np.log(p_winner))
        
        # Multiclass Brier score
        brier_scores.append(np.sum((p - y) ** 2))
        
        # Top-1 accuracy with tie handling
        max_p = np.max(p)
        top_indices = np.where(np.isclose(p, max_p, atol=1e-6))[0]
        
        if winner_idx[0] in top_indices:
            tie_aware_top1 += 1
            fractional_top1 += 1.0 / len(top_indices)
        
        if np.argmax(p) == winner_idx[0]:
            legacy_argmax_top1 += 1
            
        total_races += 1

    return {
        "races": total_races,
        "mean_log_loss": float(np.mean(log_losses)),
        "mean_brier_score": float(np.mean(brier_scores)),
        "tie_aware_top1": float(tie_aware_top1 / total_races),
        "fractional_top1": float(fractional_top1 / total_races),
        "legacy_argmax_top1": float(legacy_argmax_top1 / total_races),
    }

## 7. Model 0: Uniform Pre-Race Baseline ($1 / N_r$)

Assigns equal probability $1/N_r$ to all $N_r$ eligible pre-race entrants in each race.

In [7]:
df_m0 = df_candidates.copy()
race_sizes = df_m0.groupby("race_id")["driver_id"].transform("count")
df_m0["prob_uniform"] = 1.0 / race_sizes

# Evaluate on Full 2010-2026
res_m0_full = evaluate_race_predictions(df_m0, prob_col="prob_uniform")
print("Model 0 (Uniform Baseline) Results (Full 2010-2026):")
for k, v in res_m0_full.items():
    print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

# Evaluate on Primary Cohort (2018-2024)
res_m0_primary = evaluate_race_predictions(df_m0[PRIMARY_COHORT], prob_col="prob_uniform")
print("\nModel 0 (Uniform Baseline) Results (Primary Cohort 2018-2024):")
for k, v in res_m0_primary.items():
    print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

Model 0 (Uniform Baseline) Results (Full 2010-2026):
  races: 346
  mean_log_loss: 3.044556
  mean_brier_score: 0.952264
  tie_aware_top1: 1.000000
  fractional_top1: 0.047736
  legacy_argmax_top1: 0.014451

Model 0 (Uniform Baseline) Results (Primary Cohort 2018-2024):
  races: 149
  mean_log_loss: 2.995388
  mean_brier_score: 0.949982
  tie_aware_top1: 1.000000
  fractional_top1: 0.050018
  legacy_argmax_top1: 0.000000


## 8. Model 1: Empirical Grid-Position-Only Win Rate Baseline

Computes historical win rates conditioned strictly on starting grid position ($P(\text{Win} \mid \text{Grid})$) using a walk-forward scheme with Bayesian / Laplacian prior smoothing.

> **Smoothing Protocol:** For grid position $g$, let $W_g$ be historical wins and $T_g$ be total starts. Under Laplace smoothing with pseudocount $\alpha$:
> $$P(\text{Win} \mid g) = \frac{W_g + \alpha}{T_g + \alpha \cdot K}$$
> Probabilities for all $N_r$ entrants in race $r$ are then re-normalized across the simplex $\sum_{i=1}^{N_r} p_i = 1$.

In [8]:

# EXP-0002 / Model 1
# Walk-forward empirical grid win-frequency baseline.
#
# At each race, use only earlier race records to estimate the
# historical win frequency for each effective grid position.
# Score the current race first; update historical counts afterward.
#
# Important limitation:
# The candidates are a provisional proxy built from retrospective
# race-result records, not a verified pre-race entry list.

from collections import defaultdict
import json
import numpy as np
import pandas as pd

GRID_PRIOR_STRENGTH = 10.0  # Prespecified smoothing strength

required = [
    "race_id", "season_int", "round_int", "driver_id",
    "effective_grid", "y", "race_date",
]
missing = [c for c in required if c not in df_candidates.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

work = df_candidates.copy()
work["race_date"] = pd.to_datetime(work["race_date"], errors="coerce")
work["effective_grid"] = pd.to_numeric(
    work["effective_grid"], errors="coerce"
)
work["y"] = pd.to_numeric(work["y"], errors="coerce")

if work["race_date"].isna().any():
    raise ValueError("Missing or invalid race dates.")
if work["effective_grid"].isna().any() or (work["effective_grid"] <= 0).any():
    raise ValueError("Effective grid must contain positive numeric values.")
if not work["y"].isin([0, 1]).all():
    raise ValueError("Winner label y must be binary.")

# Ensure candidate order is independent of finishing position.
work = work.sort_values(
    ["race_date", "season_int", "round_int", "driver_id"],
    kind="stable",
).reset_index(drop=True)

assert not work.duplicated(["race_id", "driver_id"]).any()

winners_per_race = work.groupby("race_id")["y"].sum()
assert winners_per_race.eq(1).all()

# Historical candidate-record and winner counts, keyed by effective grid.
historical_records_by_grid = defaultdict(int)
historical_wins_by_grid = defaultdict(int)
historical_record_total = 0
historical_win_total = 0

prediction_parts = []
race_order_records = []

for race_id, race in work.groupby("race_id", sort=False):
    race = race.sort_values("driver_id", kind="stable").copy()
    n_candidates = len(race)

    # Prior evidence only: nothing from this race has been added yet.
    if historical_record_total == 0:
        # No prior races are available, so use the uniform distribution.
        probabilities = np.full(n_candidates, 1.0 / n_candidates)
    else:
        historical_global_rate = (
            historical_win_total / historical_record_total
        )

        rates = []
        for grid_value in race["effective_grid"]:
            grid = int(round(float(grid_value)))
            n_at_grid = historical_records_by_grid[grid]
            wins_at_grid = historical_wins_by_grid[grid]

            # Shrink sparse grid-specific estimates towards the
            # historical overall candidate win rate.
            smoothed_rate = (
                wins_at_grid
                + GRID_PRIOR_STRENGTH * historical_global_rate
            ) / (n_at_grid + GRID_PRIOR_STRENGTH)

            rates.append(smoothed_rate)

        rates = np.asarray(rates, dtype=float)
        if not np.isfinite(rates).all() or rates.sum() <= 0:
            raise ValueError(f"Invalid grid scores in race {race_id}")

        # Normalize scores to create a race-level winner distribution.
        probabilities = rates / rates.sum()

    if not np.isclose(probabilities.sum(), 1.0, atol=1e-10):
        raise AssertionError(f"Probability simplex failure in {race_id}")

    race_predictions = race[
        [
            "race_id", "race_date", "season_int", "round_int",
            "driver_id", "driver_name", "grid_position_int",
            "effective_grid", "is_grid_zero", "y",
        ]
    ].copy()

    race_predictions["prob_grid_frequency"] = probabilities
    race_predictions["historical_records_before_race"] = (
        historical_record_total
    )
    prediction_parts.append(race_predictions)

    race_order_records.append({
        "race_id": race_id,
        "season": int(race["season_int"].iloc[0]),
        "round": int(race["round_int"].iloc[0]),
        "historical_records_before_race": historical_record_total,
        "candidate_count": n_candidates,
        "probability_sum": float(probabilities.sum()),
    })

    # Only now, after generating predictions, update from the result.
    # No rows are dropped based on post-race status.
    for row in race.itertuples(index=False):
        grid = int(round(float(row.effective_grid)))
        historical_records_by_grid[grid] += 1
        historical_wins_by_grid[grid] += int(row.y)

    historical_record_total += n_candidates
    historical_win_total += int(race["y"].sum())

df_m1 = pd.concat(prediction_parts, ignore_index=True)
race_order_audit = pd.DataFrame(race_order_records)

assert len(df_m1) == len(work)
assert not df_m1.duplicated(["race_id", "driver_id"]).any()

probability_sums = df_m1.groupby("race_id")[
    "prob_grid_frequency"
].sum()

assert np.allclose(probability_sums.to_numpy(), 1.0, atol=1e-10)
assert df_m1.groupby("race_id")["y"].sum().eq(1).all()

# Evaluate the complete walk-forward series and the prespecified
# primary 2018–2024 cohort.
PRIMARY_START, PRIMARY_END = 2018, 2024
primary_mask = df_m1["season_int"].between(
    PRIMARY_START, PRIMARY_END
)

df_m1["prob_uniform"] = (
    1.0 / df_m1.groupby("race_id")["driver_id"].transform("count")
)

res_m1_full = evaluate_race_predictions(
    df_m1, prob_col="prob_grid_frequency"
)
res_m1_primary = evaluate_race_predictions(
    df_m1.loc[primary_mask], prob_col="prob_grid_frequency"
)
res_uniform_primary = evaluate_race_predictions(
    df_m1.loc[primary_mask], prob_col="prob_uniform"
)

comparison = pd.DataFrame([
    {
        "model": "Uniform baseline",
        "cohort": "2018–2024",
        "races": res_uniform_primary["races"],
        "mean_log_loss": res_uniform_primary["mean_log_loss"],
        "mean_brier_score": res_uniform_primary["mean_brier_score"],
        "tie_aware_top1": res_uniform_primary["tie_aware_top1"],
        "fractional_top1": res_uniform_primary["fractional_top1"],
    },
    {
        "model": "Empirical grid frequency",
        "cohort": "2018–2024",
        "races": res_m1_primary["races"],
        "mean_log_loss": res_m1_primary["mean_log_loss"],
        "mean_brier_score": res_m1_primary["mean_brier_score"],
        "tie_aware_top1": res_m1_primary["tie_aware_top1"],
        "fractional_top1": res_m1_primary["fractional_top1"],
    },
])

print("EXP-0002 Model 1: Walk-forward evaluation")
print("Historical grid smoothing strength:", GRID_PRIOR_STRENGTH)
print("Total race predictions:", res_m1_full["races"])
print("Primary-cohort races:", res_m1_primary["races"])
print("Candidate-level predictions:", len(df_m1))
print(
    "Maximum probability-sum error:",
    float((probability_sums - 1.0).abs().max()),
)

display(comparison)

print("\nFull-series metrics:")
display(pd.DataFrame([res_m1_full]))

# Save model-specific outputs without overwriting other experiments.
MODEL1_DIR = PROJECT_ROOT / "artifacts" / "reports" / "EXP-0002"
MODEL1_DIR.mkdir(parents=True, exist_ok=True)

predictions_path = MODEL1_DIR / "model1_grid_frequency_predictions.csv"
race_audit_path = MODEL1_DIR / "model1_walk_forward_audit.csv"
metrics_path = MODEL1_DIR / "model1_grid_frequency_metrics.json"

df_m1.to_csv(predictions_path, index=False)
race_order_audit.to_csv(race_audit_path, index=False)

manifest = {
    "experiment_id": "EXP-0002",
    "model": "Model 1: walk-forward empirical grid frequency",
    "status": "EXPLORATORY_RUN",
    "source_dataset": str(DATA_PATH.relative_to(PROJECT_ROOT)),
    "total_races": int(res_m1_full["races"]),
    "candidate_rows": int(len(df_m1)),
    "primary_cohort": [PRIMARY_START, PRIMARY_END],
    "primary_cohort_races": int(res_m1_primary["races"]),
    "grid_prior_strength": GRID_PRIOR_STRENGTH,
    "update_policy": (
        "Generate probabilities before each race; update grid "
        "frequency counts only after that race's outcome."
    ),
    "candidate_field_limitation": (
        "Candidates are provisional retrospective result-record proxies; "
        "independently timestamped pre-race entry lists are not verified "
        "for every race."
    ),
    "post_race_status_filtering": False,
    "artifacts": {
        "candidate_predictions": str(predictions_path.relative_to(PROJECT_ROOT)),
        "walk_forward_audit": str(race_audit_path.relative_to(PROJECT_ROOT)),
    },
}

metrics_path.write_text(json.dumps({
    **manifest,
    "full_series_metrics": res_m1_full,
    "primary_cohort_grid_frequency_metrics": res_m1_primary,
    "primary_cohort_uniform_metrics": res_uniform_primary,
}, indent=2), encoding="utf-8")

print("\nSaved model-specific artifacts:")
print(predictions_path)
print(race_audit_path)
print(metrics_path)

EXP-0002 Model 1: Walk-forward evaluation
Historical grid smoothing strength: 10.0
Total race predictions: 346
Primary-cohort races: 149
Candidate-level predictions: 7285
Maximum probability-sum error: 2.220446049250313e-16


,model,cohort,races,mean_log_loss,mean_brier_score,tie_aware_top1,fractional_top1
0,Uniform baseline,2018–2024,149,2.995388,0.949982,1.000000,0.050018
1,Empirical grid frequency,2018–2024,149,1.583766,0.682312,0.496644,0.496644



Full-series metrics:


,races,mean_log_loss,mean_brier_score,tie_aware_top1,fractional_top1,legacy_argmax_top1
0,346,1.519516,0.65788,0.531792,0.523242,0.523121



Saved model-specific artifacts:
/Users/dhruv/dhruv files/apex-f1/artifacts/reports/EXP-0002/model1_grid_frequency_predictions.csv
/Users/dhruv/dhruv files/apex-f1/artifacts/reports/EXP-0002/model1_walk_forward_audit.csv
/Users/dhruv/dhruv files/apex-f1/artifacts/reports/EXP-0002/model1_grid_frequency_metrics.json


## 9. Model 2: Multinomial Logit Softmax Model

Parameterizes race winner probability using a linear combination of pre-race features passed through a softmax over all entrants in race $r$:
$$s_i = \beta_1 \log(\text{effective\_grid}_i) + \beta_2 \mathbf{1}_{\text{front\_row}, i} + \beta_3 \text{constructor\_prior}_i$$
$$p_i = \frac{\exp(s_i)}{\sum_{j=1}^{N_r} \exp(s_j)}$$

This guarantees valid simplex normalization $\sum p_i = 1$ naturally without post-hoc clipping.

In [9]:
# ============================================================
# EXP-0002 — MODEL 2: WALK-FORWARD LOGISTIC REGRESSION
# ============================================================
# Protocol:
# - One prediction per candidate per race.
# - Train only on races strictly earlier than the prediction race.
# - Fixed L2 regularization; no evaluation-set hyperparameter tuning.
# - Normalize candidate win scores within each race.
# - Use uniform probabilities during the initial cold-start period.
#
# Limitation:
# Candidate rows originate from final result records, not verified
# independently timestamped pre-race entry lists.

from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# ----------------------------
# 1. Fixed experiment settings
# ----------------------------

MODEL_NAME = "Logistic regression — grid features"
PRIMARY_START_SEASON = 2018
PRIMARY_END_SEASON = 2024

MIN_HISTORY_RACES = 20
LOGISTIC_C = 1.0
MAX_ITER = 2000
RANDOM_STATE = 42

FEATURES = [
    "effective_grid",
    "log_grid",
    "is_front_row",
    "is_pole",
    "is_grid_zero",
]

OUTPUT_DIR = Path("artifacts/reports/EXP-0002")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ----------------------------
# 2. Validate and prepare data
# ----------------------------

required_columns = [
    "race_id",
    "race_date",
    "season_int",
    "round_int",
    "driver_id",
    "driver_name",
    "y",
    *FEATURES,
]

missing_columns = sorted(
    set(required_columns) - set(df_candidates.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing required columns in df_candidates: {missing_columns}"
    )

data = df_candidates[required_columns].copy()

if data[required_columns].isna().any().any():
    missing_counts = data[required_columns].isna().sum()
    raise ValueError(
        "Missing values detected. Investigate rather than silently "
        f"imputing them:\n{missing_counts[missing_counts > 0]}"
    )

data["race_date"] = pd.to_datetime(data["race_date"])
data["season_int"] = data["season_int"].astype(int)
data["round_int"] = data["round_int"].astype(int)
data["y"] = data["y"].astype(int)

if not data["y"].isin([0, 1]).all():
    raise ValueError("Target y must contain only 0 and 1.")

if not data[FEATURES].apply(
    lambda col: pd.to_numeric(col, errors="coerce").notna().all()
).all():
    raise ValueError("One or more model features are not numeric.")

data[FEATURES] = data[FEATURES].astype(float)

if not np.isfinite(data[FEATURES].to_numpy()).all():
    raise ValueError("Non-finite feature values found.")

if data.duplicated(["race_id", "driver_id"]).any():
    raise ValueError("Duplicate race_id/driver_id candidate rows found.")

# There must be exactly one recorded winner per race.
winner_counts = data.groupby("race_id")["y"].sum()

if not winner_counts.eq(1).all():
    raise ValueError(
        "Expected exactly one recorded winner in every race. "
        f"Invalid race count: {(winner_counts != 1).sum()}"
    )

# Race-level chronological ordering, independent of finishing position.
race_table = (
    data.groupby("race_id", as_index=False)
    .agg(
        race_date=("race_date", "min"),
        season_int=("season_int", "first"),
        round_int=("round_int", "first"),
    )
    .sort_values(
        ["race_date", "season_int", "round_int", "race_id"]
    )
    .reset_index(drop=True)
)

if len(race_table) != data["race_id"].nunique():
    raise ValueError("Race chronology contains unexpected duplicate IDs.")

data = data.sort_values(
    ["race_date", "season_int", "round_int", "driver_id"]
).reset_index(drop=True)


# ----------------------------
# 3. Walk-forward predictions
# ----------------------------

prediction_records = []
audit_records = []

for race_index, race in race_table.iterrows():
    current_race_id = race["race_id"]
    current = data.loc[data["race_id"] == current_race_id].copy()

    # Critical leakage safeguard: train only on earlier race IDs.
    earlier_race_ids = race_table.iloc[:race_index]["race_id"].tolist()
    train = data.loc[data["race_id"].isin(earlier_race_ids)].copy()

    n_train_races = len(earlier_race_ids)
    used_fallback = n_train_races < MIN_HISTORY_RACES

    if used_fallback:
        probabilities = np.full(
            len(current),
            1.0 / len(current),
            dtype=float,
        )
        prediction_method = "uniform_cold_start"

    else:
        model = Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "logistic",
                    LogisticRegression(
                        penalty="l2",
                        C=LOGISTIC_C,
                        solver="lbfgs",
                        max_iter=MAX_ITER,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        )

        # The model sees only prior races' candidate features and outcomes.
        model.fit(train[FEATURES], train["y"])

        raw_scores = model.predict_proba(current[FEATURES])[:, 1]

        # Convert independent binary scores into a race-level distribution.
        raw_scores = np.clip(raw_scores, 1e-12, None)
        probabilities = raw_scores / raw_scores.sum()
        prediction_method = "walk_forward_logistic"

    for row_index, (_, candidate) in enumerate(current.iterrows()):
        prediction_records.append(
            {
                "model": "model2_logistic_grid",
                "race_id": candidate["race_id"],
                "race_date": candidate["race_date"],
                "season_int": int(candidate["season_int"]),
                "round_int": int(candidate["round_int"]),
                "driver_id": candidate["driver_id"],
                "driver_name": candidate["driver_name"],
                "y": int(candidate["y"]),
                "probability": float(probabilities[row_index]),
                "prediction_method": prediction_method,
                "n_train_races": n_train_races,
            }
        )

    audit_records.append(
        {
            "race_id": current_race_id,
            "race_date": race["race_date"],
            "season_int": int(race["season_int"]),
            "round_int": int(race["round_int"]),
            "n_candidates": len(current),
            "n_train_races": n_train_races,
            "used_uniform_cold_start": used_fallback,
            "prediction_method": prediction_method,
            "probability_sum": float(probabilities.sum()),
        }
    )

model2_predictions = pd.DataFrame(prediction_records)
model2_audit = pd.DataFrame(audit_records)

# Structural checks.
assert len(model2_predictions) == len(data)
assert model2_predictions.groupby("race_id").size().equals(
    data.groupby("race_id").size()
)
assert np.allclose(
    model2_predictions.groupby("race_id")["probability"].sum().to_numpy(),
    1.0,
    atol=1e-10,
)

print("EXP-0002 Model 2: walk-forward logistic regression")
print(f"Races predicted: {model2_predictions['race_id'].nunique()}")
print(f"Candidate rows: {len(model2_predictions)}")
print(f"Cold-start races: {model2_audit['used_uniform_cold_start'].sum()}")
print(
    "Maximum probability-sum error:",
    float(
        (
            model2_predictions.groupby("race_id")["probability"].sum() - 1.0
        ).abs().max()
    ),
)


# ----------------------------
# 4. Race-level evaluation
# ----------------------------

def evaluate_model2_predictions(predictions):
    """Evaluate one-winner race probability distributions."""

    race_log_losses = []
    race_brier_scores = []
    fractional_top1_scores = []
    tie_aware_top1_scores = []

    for _, race_group in predictions.groupby("race_id", sort=False):
        y_true = race_group["y"].to_numpy(dtype=int)
        probs = race_group["probability"].to_numpy(dtype=float)

        if y_true.sum() != 1:
            raise ValueError("Evaluation race does not have exactly one winner.")

        if not np.isclose(probs.sum(), 1.0, atol=1e-10):
            raise ValueError("Race probabilities do not sum to one.")

        winner_index = int(np.flatnonzero(y_true == 1)[0])
        race_log_losses.append(
            -np.log(np.clip(probs[winner_index], 1e-15, 1.0))
        )

        # Multiclass Brier score: sum across candidates, average across races.
        race_brier_scores.append(float(np.sum((probs - y_true) ** 2)))

        max_probability = probs.max()
        top_mask = np.isclose(
            probs, max_probability, rtol=1e-9, atol=1e-12
        )
        winner_is_top = bool(top_mask[winner_index])

        # Fractional credit for a winner tied among k top candidates.
        fractional_top1_scores.append(
            float(top_mask[winner_index] / top_mask.sum())
        )
        tie_aware_top1_scores.append(float(winner_is_top))

    return {
        "races": len(race_log_losses),
        "mean_log_loss": float(np.mean(race_log_losses)),
        "mean_brier_score": float(np.mean(race_brier_scores)),
        "fractional_top1": float(np.mean(fractional_top1_scores)),
        "tie_aware_top1": float(np.mean(tie_aware_top1_scores)),
    }


full_metrics = evaluate_model2_predictions(model2_predictions)

primary_predictions = model2_predictions.loc[
    model2_predictions["season_int"].between(
        PRIMARY_START_SEASON, PRIMARY_END_SEASON
    )
].copy()

primary_metrics = evaluate_model2_predictions(primary_predictions)

model2_metrics = {
    "experiment": "EXP-0002",
    "model": MODEL_NAME,
    "features": FEATURES,
    "min_history_races": MIN_HISTORY_RACES,
    "logistic_C": LOGISTIC_C,
    "max_iter": MAX_ITER,
    "random_state": RANDOM_STATE,
    "primary_cohort": f"{PRIMARY_START_SEASON}-{PRIMARY_END_SEASON}",
    "full_series": full_metrics,
    "primary_cohort_metrics": primary_metrics,
    "candidate_set_limitation": (
        "Candidates are a provisional proxy from final result records; "
        "independently timestamped pre-race entry lists are not verified for every race."
    ),
    "artifacts": {
        "candidate_predictions": "artifacts/reports/EXP-0002/model2_logistic_predictions.csv",
        "walk_forward_audit": "artifacts/reports/EXP-0002/model2_walk_forward_audit.csv",
    },
}

predictions_path = OUTPUT_DIR / "model2_logistic_predictions.csv"
audit_path = OUTPUT_DIR / "model2_walk_forward_audit.csv"
metrics_path = OUTPUT_DIR / "model2_logistic_metrics.json"

model2_predictions.to_csv(predictions_path, index=False)
model2_audit.to_csv(audit_path, index=False)
metrics_path.write_text(json.dumps(model2_metrics, indent=2), encoding="utf-8")

print("\nSaved Model 2 artifacts:")
print(predictions_path)
print(audit_path)
print(metrics_path)

print("\nModel 2 Primary Cohort Metrics (2018–2024):")
for k, v in primary_metrics.items():
    print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

print("\nModel 2 Full Series Metrics (2010–2026):")
for k, v in full_metrics.items():
    print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")


EXP-0002 Model 2: walk-forward logistic regression
Races predicted: 346
Candidate rows: 7285
Cold-start races: 20
Maximum probability-sum error: 4.440892098500626e-16

Saved Model 2 artifacts:
artifacts/reports/EXP-0002/model2_logistic_predictions.csv
artifacts/reports/EXP-0002/model2_walk_forward_audit.csv
artifacts/reports/EXP-0002/model2_logistic_metrics.json

Model 2 Primary Cohort Metrics (2018–2024):
  races: 149
  mean_log_loss: 1.594207
  mean_brier_score: 0.683382
  fractional_top1: 0.496644
  tie_aware_top1: 0.496644

Model 2 Full Series Metrics (2010–2026):
  races: 346
  mean_log_loss: 1.541942
  mean_brier_score: 0.662375
  fractional_top1: 0.508200
  tie_aware_top1: 0.563584


## 10. Model 3: Grid + Driver and Constructor Form Logistic Regression

Expands the static starting grid benchmark by incorporating rolling historical driver and constructor form features available strictly prior to each race weekend:
- **Grid Features (5):** `effective_grid`, `log_grid`, `is_front_row`, `is_pole`, `is_grid_zero`
- **Driver Form Features (5):** Prior starts count, career win rate, last-5 win rate, last-5 mean finish, last-5 finish std
- **Constructor Form Features (4):** Prior starts count, career win rate, last-5 win rate, last-5 mean finish
- **Points Form Features (2):** Driver and constructor last-5 mean points

Trained under a strict walk-forward protocol with L2 regularization ($C=1.0$) and a 20-race uniform cold-start period.


In [10]:
# ============================================================
# EXP-0002 — MODEL 3: GRID + DRIVER FORM + CONSTRUCTOR FORM
# Strict walk-forward evaluation; no same-race history updates.
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# ------------------------------------------------------------
# 1. Fixed settings
# ------------------------------------------------------------

OUTPUT_DIR = Path("artifacts/reports/EXP-0002")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_HISTORY_RACES = 20
LOGISTIC_C = 1.0
MAX_ITER = 2000
RANDOM_STATE = 42

GRID_FEATURES = [
    "effective_grid",
    "log_grid",
    "is_front_row",
    "is_pole",
    "is_grid_zero",
]

# Identify the actual columns present in this dataset.
def find_column(candidates, columns):
    return next((c for c in candidates if c in columns), None)


constructor_col = find_column(
    ["constructor_id", "constructor_ref", "constructor_name", "constructor"],
    df_candidates.columns,
)

finish_col = find_column(
    [
        "position_order",
        "positionOrder",
        "position_order_int",
        "position_int",
        "finish_position",
        "position",
    ],
    df_candidates.columns,
)

points_col = find_column(
    ["points", "points_num", "points_float"],
    df_candidates.columns,
)

if constructor_col is None:
    raise ValueError(
        "Model 3 requires a constructor identifier. Could not find one of "
        "constructor_id, constructor_ref, constructor_name, constructor. "
        f"Available columns: {df_candidates.columns.tolist()}"
    )

if finish_col is None:
    raise ValueError(
        "Model 3 requires a historical finishing-position field to calculate "
        "driver and constructor form. Could not identify one. "
        f"Available columns: {df_candidates.columns.tolist()}"
    )

required = [
    "race_id", "race_date", "season_int", "round_int",
    "driver_id", "driver_name", "y", *GRID_FEATURES,
    constructor_col, finish_col,
]
if points_col is not None:
    required.append(points_col)

missing = [c for c in required if c not in df_candidates.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

data = df_candidates[list(dict.fromkeys(required))].copy()
data["race_date"] = pd.to_datetime(data["race_date"])
data["season_int"] = data["season_int"].astype(int)
data["round_int"] = data["round_int"].astype(int)
data["y"] = pd.to_numeric(data["y"], errors="raise").astype(int)

if not data["y"].isin([0, 1]).all():
    raise ValueError("Target y must contain only 0 and 1.")

for col in GRID_FEATURES:
    data[col] = pd.to_numeric(data[col], errors="raise")

data["_finish"] = pd.to_numeric(data[finish_col], errors="coerce")
data.loc[data["_finish"] <= 0, "_finish"] = np.nan

if points_col is not None:
    data["_points"] = pd.to_numeric(data[points_col], errors="coerce")
    data["_points"] = data["_points"].fillna(0.0)
else:
    data["_points"] = 0.0

if data.duplicated(["race_id", "driver_id"]).any():
    raise ValueError("Duplicate race_id/driver_id candidate rows found.")

winner_counts = data.groupby("race_id")["y"].sum()
if not winner_counts.eq(1).all():
    raise ValueError("Expected exactly one recorded winner in every race.")

race_table = (
    data.groupby("race_id", as_index=False)
    .agg(
        race_date=("race_date", "min"),
        season_int=("season_int", "first"),
        round_int=("round_int", "first"),
    )
    .sort_values(["race_date", "season_int", "round_int", "race_id"])
    .reset_index(drop=True)
)

if len(race_table) != data["race_id"].nunique():
    raise ValueError("Unexpected duplicate race IDs in chronological table.")

print("Constructor column:", constructor_col)
print("Historical finish column:", finish_col)
print("Historical points column:", points_col or "Unavailable; points features disabled")


# ------------------------------------------------------------
# 2. Define features
# ------------------------------------------------------------

FORM_FEATURES = [
    "driver_prior_races",
    "driver_win_rate_all",
    "driver_win_rate_last5",
    "driver_mean_finish_last5",
    "driver_finish_std_last5",
    "constructor_prior_races",
    "constructor_win_rate_all",
    "constructor_win_rate_last5",
    "constructor_mean_finish_last5",
]

if points_col is not None:
    FORM_FEATURES += [
        "driver_mean_points_last5",
        "constructor_mean_points_last5",
    ]

MODEL_FEATURES = GRID_FEATURES + FORM_FEATURES


def valid_values(records, key, window=None):
    """Return finite history values, considering only the requested tail."""
    subset = records if window is None else records[-window:]
    out = []
    for record in subset:
        value = record.get(key)
        if value is not None and np.isfinite(value):
            out.append(float(value))
    return out


def mean_or(values, fallback=0.0):
    return float(np.mean(values)) if values else float(fallback)


def sample_std_or_zero(values):
    return float(np.std(values, ddof=1)) if len(values) >= 2 else 0.0


# The history dictionaries are updated only AFTER the entire race is predicted.
driver_history = {}
constructor_history = {}

feature_rows = []
prediction_rows = []
audit_rows = []


# ------------------------------------------------------------
# 3. Walk forward one race at a time
# ------------------------------------------------------------

for race_index, race_info in race_table.iterrows():
    race_id = race_info["race_id"]
    current = (
        data.loc[data["race_id"] == race_id]
        .sort_values("driver_id")
        .copy()
    )

    # Build every candidate's features using the state BEFORE this race.
    current_feature_rows = []

    for _, row in current.iterrows():
        driver_key = str(row["driver_id"])
        constructor_value = row[constructor_col]
        constructor_key = (
            None if pd.isna(constructor_value) else str(constructor_value)
        )

        dh = driver_history.get(driver_key, [])
        ch = (
            constructor_history.get(constructor_key, [])
            if constructor_key is not None else []
        )

        driver_recent = dh[-5:]
        constructor_recent = ch[-5:]

        driver_finishes = valid_values(dh, "finish", window=5)
        constructor_finishes = valid_values(ch, "mean_finish", window=5)

        features = {feature: float(row[feature]) for feature in GRID_FEATURES}

        features.update({
            "driver_prior_races": float(len(dh)),
            "driver_win_rate_all": mean_or(
                valid_values(dh, "win"), fallback=0.0
            ),
            "driver_win_rate_last5": mean_or(
                valid_values(driver_recent, "win"), fallback=0.0
            ),
            # Fixed cold-start value, not estimated from future races.
            "driver_mean_finish_last5": mean_or(
                driver_finishes, fallback=20.0
            ),
            "driver_finish_std_last5": sample_std_or_zero(driver_finishes),
            "constructor_prior_races": float(len(ch)),
            "constructor_win_rate_all": mean_or(
                valid_values(ch, "win"), fallback=0.0
            ),
            "constructor_win_rate_last5": mean_or(
                valid_values(constructor_recent, "win"), fallback=0.0
            ),
            "constructor_mean_finish_last5": mean_or(
                constructor_finishes, fallback=20.0
            ),
        })

        if points_col is not None:
            features["driver_mean_points_last5"] = mean_or(
                valid_values(dh, "points", window=5), fallback=0.0
            )
            features["constructor_mean_points_last5"] = mean_or(
                valid_values(ch, "total_points", window=5), fallback=0.0
            )

        record = {
            "race_id": row["race_id"],
            "race_date": row["race_date"],
            "season_int": int(row["season_int"]),
            "round_int": int(row["round_int"]),
            "driver_id": row["driver_id"],
            "driver_name": row["driver_name"],
            "constructor_key": constructor_key,
            "y": int(row["y"]),
            **features,
        }

        current_feature_rows.append(record)

    current_features = pd.DataFrame(current_feature_rows)

    # At this point feature_rows contains only earlier races.
    history_frame = pd.DataFrame(feature_rows)
    n_train_races = race_index
    use_fallback = (
        n_train_races < MIN_HISTORY_RACES
        or history_frame.empty
        or history_frame["y"].nunique() < 2
    )

    if use_fallback:
        probabilities = np.full(len(current_features), 1.0 / len(current_features))
        method = "uniform_cold_start"
    else:
        model = Pipeline([
            ("scaler", StandardScaler()),
            (
                "logistic",
                LogisticRegression(
                    penalty="l2",
                    C=LOGISTIC_C,
                    solver="lbfgs",
                    max_iter=MAX_ITER,
                    random_state=RANDOM_STATE,
                ),
            ),
        ])

        model.fit(
            history_frame[MODEL_FEATURES].astype(float),
            history_frame["y"].astype(int),
        )

        class_one_index = list(model.named_steps["logistic"].classes_).index(1)
        raw_scores = model.predict_proba(
            current_features[MODEL_FEATURES].astype(float)
        )[:, class_one_index]

        raw_scores = np.clip(raw_scores, 1e-12, None)
        probabilities = raw_scores / raw_scores.sum()
        method = "walk_forward_logistic_form"

    # Store predictions and the features used for this race.
    current_features["probability"] = probabilities
    current_features["prediction_method"] = method
    current_features["n_train_races"] = n_train_races

    feature_rows.extend(current_feature_rows)

    prediction_rows.extend(
        current_features[
            [
                "race_id", "race_date", "season_int", "round_int",
                "driver_id", "driver_name", "y", "probability",
                "prediction_method", "n_train_races",
            ]
        ].to_dict(orient="records")
    )

    audit_rows.append({
        "race_id": race_id,
        "race_date": race_info["race_date"],
        "season_int": int(race_info["season_int"]),
        "round_int": int(race_info["round_int"]),
        "n_candidates": len(current_features),
        "n_train_races": n_train_races,
        "used_uniform_cold_start": use_fallback,
        "prediction_method": method,
        "probability_sum": float(probabilities.sum()),
    })

    # IMPORTANT: update history only after all candidates in this race
    # have been predicted. No driver or constructor can see same-race outcomes.
    for _, row in current.iterrows():
        driver_key = str(row["driver_id"])
        finish = row["_finish"]
        points = row["_points"]

        driver_history.setdefault(driver_key, []).append({
            "win": int(row["y"]),
            "finish": float(finish) if pd.notna(finish) else None,
            "points": float(points) if pd.notna(points) else 0.0,
        })

    # One history record per constructor per Grand Prix weekend, not per driver.
    current_with_constructor = current.loc[
        current[constructor_col].notna()
    ].copy()

    for constructor_value, group in current_with_constructor.groupby(
        constructor_col, sort=False
    ):
        constructor_key = str(constructor_value)
        finishes = group["_finish"].dropna().astype(float).tolist()

        constructor_history.setdefault(constructor_key, []).append({
            "win": int(group["y"].max()),
            "mean_finish": float(np.mean(finishes)) if finishes else None,
            "total_points": float(group["_points"].sum()),
        })


# ------------------------------------------------------------
# 4. Validate predictions and evaluate
# ------------------------------------------------------------

model3_predictions = pd.DataFrame(prediction_rows)
model3_audit = pd.DataFrame(audit_rows)

assert len(model3_predictions) == len(data)
assert model3_predictions["race_id"].nunique() == len(race_table)

probability_sums = model3_predictions.groupby("race_id")["probability"].sum()
assert np.allclose(probability_sums.to_numpy(), 1.0, atol=1e-10)

assert model3_predictions["probability"].between(0, 1).all()

def evaluate_model3_predictions(predictions):
    log_losses = []
    brier_scores = []
    fractional_top1 = []
    tie_aware_top1 = []

    for _, race_group in predictions.groupby("race_id", sort=False):
        y = race_group["y"].to_numpy(dtype=int)
        p = race_group["probability"].to_numpy(dtype=float)

        if y.sum() != 1:
            raise ValueError("Evaluation requires exactly one recorded winner per race.")
        if not np.isclose(p.sum(), 1.0, atol=1e-10):
            raise ValueError("Predicted race probabilities do not sum to one.")

        winner_index = int(np.flatnonzero(y == 1)[0])
        log_losses.append(-np.log(np.clip(p[winner_index], 1e-15, 1.0)))
        brier_scores.append(float(np.sum((p - y) ** 2)))

        top_mask = np.isclose(p, p.max(), rtol=1e-9, atol=1e-12)
        fractional_top1.append(
            float(top_mask[winner_index] / top_mask.sum())
        )
        tie_aware_top1.append(float(top_mask[winner_index]))

    return {
        "races": len(log_losses),
        "mean_log_loss": float(np.mean(log_losses)),
        "mean_brier_score": float(np.mean(brier_scores)),
        "fractional_top1": float(np.mean(fractional_top1)),
        "tie_aware_top1": float(np.mean(tie_aware_top1)),
    }


full_metrics = evaluate_model3_predictions(model3_predictions)

primary_predictions = model3_predictions.loc[
    model3_predictions["season_int"].between(2018, 2024)
].copy()
primary_metrics = evaluate_model3_predictions(primary_predictions)

model3_metrics = {
    "experiment": "EXP-0002",
    "model": "Model 3 — Grid + driver and constructor form",
    "constructor_column": constructor_col,
    "finish_column": finish_col,
    "points_column": points_col,
    "features": MODEL_FEATURES,
    "min_history_races": MIN_HISTORY_RACES,
    "logistic_C": LOGISTIC_C,
    "max_iter": MAX_ITER,
    "random_state": RANDOM_STATE,
    "primary_cohort": "2018-2024",
    "full_series": full_metrics,
    "primary_cohort_metrics": primary_metrics,
    "candidate_set_limitation": (
        "Candidate records originate from final result records; independently "
        "timestamped pre-race entry lists have not been verified."
    ),
    "probability_interpretation": (
        "Binary logistic scores normalized across candidates within each race; "
        "not assumed to be calibrated conditional winner probabilities."
    ),
    "points_feature_note": (
        "Historical points features were included."
        if points_col is not None
        else "No points column was found; points features were omitted."
    ),
}

model3_predictions.to_csv(
    OUTPUT_DIR / "model3_driver_constructor_predictions.csv",
    index=False,
)
model3_audit.to_csv(
    OUTPUT_DIR / "model3_walk_forward_audit.csv",
    index=False,
)
with open(
    OUTPUT_DIR / "model3_metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(model3_metrics, file, indent=2, default=str)

print("EXP-0002 Model 3: walk-forward driver + constructor form")
print("Races predicted:", model3_predictions["race_id"].nunique())
print("Candidate rows:", len(model3_predictions))
print("Cold-start races:", int(model3_audit["used_uniform_cold_start"].sum()))
print("Maximum probability-sum error:", float((probability_sums - 1).abs().max()))

print("\nModel 3 — Primary cohort (2018–2024)")
display(pd.DataFrame([primary_metrics]))

print("\nModel 3 — Full series (2010–2026)")
display(pd.DataFrame([full_metrics]))

print("\nExperiment artifacts saved to:", OUTPUT_DIR.resolve())

Constructor column: constructor_id
Historical finish column: finish_position
Historical points column: points
EXP-0002 Model 3: walk-forward driver + constructor form
Races predicted: 346
Candidate rows: 7285
Cold-start races: 20
Maximum probability-sum error: 3.3306690738754696e-16

Model 3 — Primary cohort (2018–2024)


,races,mean_log_loss,mean_brier_score,fractional_top1,tie_aware_top1
0,149,1.355201,0.612698,0.516779,0.516779



Model 3 — Full series (2010–2026)


,races,mean_log_loss,mean_brier_score,fractional_top1,tie_aware_top1
0,346,1.419383,0.624583,0.525541,0.580925



Experiment artifacts saved to: /Users/dhruv/dhruv files/apex-f1/artifacts/reports/EXP-0002


## 11. Model 4: Tree-Based Pre-Race GBDT Benchmark (LightGBM)

Evaluates whether a nonlinear gradient boosted decision tree classifier (LightGBM) can improve out-of-sample probabilistic race-winner forecasting over linear/logistic form models.

### Model Architecture & Configuration
Uses the exact same 16 pre-race features as Model 3 (5 grid features + 11 driver/constructor form features) under a strict walk-forward evaluation protocol.

**Fixed Initial Hyperparameters (Prespecified, no test-set tuning):**
- `objective`: `'binary'`
- `n_estimators`: 250
- `learning_rate`: 0.03
- `num_leaves`: 15
- `max_depth`: -1
- `min_child_samples`: 20
- `subsample`: 0.8
- `colsample_bytree`: 0.8
- `reg_lambda`: 1.0
- `random_state`: 42
- `n_jobs`: 1

> **Probability Interpretation Note:** Positive-class binary scores are normalized across all entrants in each Grand Prix ($\sum_{i=1}^{N_r} p_i = 1$). Like Model 2 and Model 3, normalized binary classifier scores provide relative ranking and comparative calibration, but are not assumed to be intrinsically calibrated conditional multinomial probabilities.


In [11]:
# ============================================================
# EXP-0002 — MODEL 4: LIGHTGBM GRADIENT BOOSTED TREES
# Strict walk-forward evaluation; identical features to Model 3.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import lightgbm as lgb

# ------------------------------------------------------------
# 1. Fixed experiment settings
# ------------------------------------------------------------

MODEL_NAME = "Model 4 — LightGBM GBDT (grid + driver/constructor form)"
OUTPUT_DIR = Path("artifacts/reports/EXP-0002")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_HISTORY_RACES = 20
RANDOM_STATE = 42

LGB_PARAMS = {
    "objective": "binary",
    "n_estimators": 250,
    "learning_rate": 0.03,
    "num_leaves": 15,
    "max_depth": -1,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
    "verbose": -1,
}

# Verify feature consistency with Model 3
assert "MODEL_FEATURES" in globals(), "MODEL_FEATURES from Model 3 required."
print(f"LightGBM feature count: {len(MODEL_FEATURES)}")
print("Features:", MODEL_FEATURES)

# ------------------------------------------------------------
# 2. Chronological Walk-Forward Training & Inference
# ------------------------------------------------------------

driver_history = {}
constructor_history = {}
feature_rows = []
prediction_rows = []
audit_rows = []

print("\nRunning Model 4 walk-forward evaluation across all races...")
for race_index, race_info in race_table.iterrows():
    race_id = race_info["race_id"]
    current = (
        data.loc[data["race_id"] == race_id]
        .sort_values("driver_id")
        .copy()
    )

    # Compute features for current race candidates using strictly prior evidence
    current_feature_rows = []
    for _, row in current.iterrows():
        driver_key = str(row["driver_id"])
        constructor_value = row[constructor_col]
        constructor_key = (
            None if pd.isna(constructor_value) else str(constructor_value)
        )

        dh = driver_history.get(driver_key, [])
        ch = (
            constructor_history.get(constructor_key, [])
            if constructor_key is not None else []
        )

        driver_recent = dh[-5:]
        constructor_recent = ch[-5:]

        driver_finishes = valid_values(dh, "finish", window=5)
        constructor_finishes = valid_values(ch, "mean_finish", window=5)

        features = {feature: float(row[feature]) for feature in GRID_FEATURES}
        features.update({
            "driver_prior_races": float(len(dh)),
            "driver_win_rate_all": mean_or(valid_values(dh, "win"), fallback=0.0),
            "driver_win_rate_last5": mean_or(valid_values(driver_recent, "win"), fallback=0.0),
            "driver_mean_finish_last5": mean_or(driver_finishes, fallback=20.0),
            "driver_finish_std_last5": sample_std_or_zero(driver_finishes),
            "constructor_prior_races": float(len(ch)),
            "constructor_win_rate_all": mean_or(valid_values(ch, "win"), fallback=0.0),
            "constructor_win_rate_last5": mean_or(valid_values(constructor_recent, "win"), fallback=0.0),
            "constructor_mean_finish_last5": mean_or(constructor_finishes, fallback=20.0),
        })

        if points_col is not None:
            features["driver_mean_points_last5"] = mean_or(valid_values(dh, "points", window=5), fallback=0.0)
            features["constructor_mean_points_last5"] = mean_or(valid_values(ch, "total_points", window=5), fallback=0.0)

        record = {
            "race_id": row["race_id"],
            "race_date": row["race_date"],
            "season_int": int(row["season_int"]),
            "round_int": int(row["round_int"]),
            "driver_id": row["driver_id"],
            "driver_name": row["driver_name"],
            "constructor_key": constructor_key,
            "y": int(row["y"]),
            **features,
        }
        current_feature_rows.append(record)

    current_features = pd.DataFrame(current_feature_rows)
    history_frame = pd.DataFrame(feature_rows)
    n_train_races = race_index
    use_fallback = (
        n_train_races < MIN_HISTORY_RACES
        or history_frame.empty
        or history_frame["y"].nunique() < 2
    )

    if use_fallback:
        probabilities = np.full(len(current_features), 1.0 / len(current_features))
        method = "uniform_cold_start"
    else:
        model = lgb.LGBMClassifier(**LGB_PARAMS)
        model.fit(
            history_frame[MODEL_FEATURES].astype(float),
            history_frame["y"].astype(int),
        )
        class_one_index = list(model.classes_).index(1)
        raw_scores = model.predict_proba(
            current_features[MODEL_FEATURES].astype(float)
        )[:, class_one_index]
        raw_scores = np.clip(raw_scores, 1e-12, None)
        probabilities = raw_scores / raw_scores.sum()
        method = "walk_forward_lightgbm"

    current_features["probability"] = probabilities
    current_features["prediction_method"] = method
    current_features["n_train_races"] = n_train_races

    feature_rows.extend(current_feature_rows)
    prediction_rows.extend(
        current_features[
            [
                "race_id", "race_date", "season_int", "round_int",
                "driver_id", "driver_name", "y", "probability",
                "prediction_method", "n_train_races",
            ]
        ].to_dict(orient="records")
    )
    audit_rows.append({
        "race_id": race_id,
        "race_date": race_info["race_date"],
        "season_int": int(race_info["season_int"]),
        "round_int": int(race_info["round_int"]),
        "n_candidates": len(current_features),
        "n_train_races": n_train_races,
        "used_uniform_cold_start": use_fallback,
        "prediction_method": method,
        "probability_sum": float(probabilities.sum()),
    })

    # Update driver and constructor histories only after prediction
    for _, row in current.iterrows():
        driver_key = str(row["driver_id"])
        finish = row["_finish"]
        points = row["_points"]
        driver_history.setdefault(driver_key, []).append({
            "win": int(row["y"]),
            "finish": float(finish) if pd.notna(finish) else None,
            "points": float(points) if pd.notna(points) else 0.0,
        })

    current_with_constructor = current.loc[current[constructor_col].notna()].copy()
    for constructor_value, group in current_with_constructor.groupby(constructor_col, sort=False):
        constructor_key = str(constructor_value)
        finishes = group["_finish"].dropna().astype(float).tolist()
        constructor_history.setdefault(constructor_key, []).append({
            "win": int(group["y"].max()),
            "mean_finish": float(np.mean(finishes)) if finishes else None,
            "total_points": float(group["_points"].sum()),
        })

# ------------------------------------------------------------
# 3. Validation & Metrics Computation
# ------------------------------------------------------------

model4_predictions = pd.DataFrame(prediction_rows)
model4_audit = pd.DataFrame(audit_rows)

assert len(model4_predictions) == len(data), "Mismatch in candidate prediction row count!"
assert model4_predictions["race_id"].nunique() == len(race_table), "Mismatch in unique race count!"

prob_sums = model4_predictions.groupby("race_id")["probability"].sum()
assert np.allclose(prob_sums.to_numpy(), 1.0, atol=1e-10), "Probability simplex violation!"
assert model4_predictions["probability"].between(0, 1).all(), "Probability bounds violation!"

full_metrics_m4 = evaluate_model3_predictions(model4_predictions)
primary_predictions_m4 = model4_predictions.loc[
    model4_predictions["season_int"].between(2018, 2024)
].copy()
primary_metrics_m4 = evaluate_model3_predictions(primary_predictions_m4)

# ------------------------------------------------------------
# 4. Persist Independent Model 4 Artifacts
# ------------------------------------------------------------

model4_metrics = {
    "experiment": "EXP-0002",
    "model": MODEL_NAME,
    "algorithm": "LightGBM GBDT (LGBMClassifier)",
    "hyperparameters": LGB_PARAMS,
    "features": MODEL_FEATURES,
    "feature_count": len(MODEL_FEATURES),
    "min_history_races": MIN_HISTORY_RACES,
    "random_state": RANDOM_STATE,
    "primary_cohort": "2018-2024",
    "full_series": full_metrics_m4,
    "primary_cohort_metrics": primary_metrics_m4,
    "candidate_set_limitation": (
        "Candidate records originate from final result records; independently "
        "timestamped pre-race entry lists have not been verified."
    ),
    "probability_interpretation": (
        "Binary GBDT scores normalized across candidates within each race; "
        "not assumed to be calibrated conditional winner probabilities."
    ),
    "artifacts": {
        "candidate_predictions": "artifacts/reports/EXP-0002/model4_lightgbm_predictions.csv",
        "walk_forward_audit": "artifacts/reports/EXP-0002/model4_walk_forward_audit.csv",
    },
}

preds_path_m4 = OUTPUT_DIR / "model4_lightgbm_predictions.csv"
audit_path_m4 = OUTPUT_DIR / "model4_walk_forward_audit.csv"
metrics_path_m4 = OUTPUT_DIR / "model4_metrics.json"

model4_predictions.to_csv(preds_path_m4, index=False)
model4_audit.to_csv(audit_path_m4, index=False)
with open(metrics_path_m4, "w", encoding="utf-8") as f:
    json.dump(model4_metrics, f, indent=2, default=str)

print("EXP-0002 Model 4: Walk-Forward LightGBM GBDT")
print("Races predicted:", model4_predictions["race_id"].nunique())
print("Candidate rows:", len(model4_predictions))
print("Cold-start races:", int(model4_audit["used_uniform_cold_start"].sum()))
print("Maximum probability-sum error:", float((prob_sums - 1.0).abs().max()))

print("\nModel 4 — Primary cohort (2018–2024):")
display(pd.DataFrame([primary_metrics_m4]))

print("\nModel 4 — Full series (2010–2026):")
display(pd.DataFrame([full_metrics_m4]))

print("\nSaved Model 4 artifacts:")
print(preds_path_m4)
print(audit_path_m4)
print(metrics_path_m4)


LightGBM feature count: 16
Features: ['effective_grid', 'log_grid', 'is_front_row', 'is_pole', 'is_grid_zero', 'driver_prior_races', 'driver_win_rate_all', 'driver_win_rate_last5', 'driver_mean_finish_last5', 'driver_finish_std_last5', 'constructor_prior_races', 'constructor_win_rate_all', 'constructor_win_rate_last5', 'constructor_mean_finish_last5', 'driver_mean_points_last5', 'constructor_mean_points_last5']

Running Model 4 walk-forward evaluation across all races...
EXP-0002 Model 4: Walk-Forward LightGBM GBDT
Races predicted: 346
Candidate rows: 7285
Cold-start races: 20
Maximum probability-sum error: 3.3306690738754696e-16

Model 4 — Primary cohort (2018–2024):


,races,mean_log_loss,mean_brier_score,fractional_top1,tie_aware_top1
0,149,1.409149,0.628561,0.530201,0.530201



Model 4 — Full series (2010–2026):


,races,mean_log_loss,mean_brier_score,fractional_top1,tie_aware_top1
0,346,1.52265,0.650861,0.51687,0.572254



Saved Model 4 artifacts:
artifacts/reports/EXP-0002/model4_lightgbm_predictions.csv
artifacts/reports/EXP-0002/model4_walk_forward_audit.csv
artifacts/reports/EXP-0002/model4_metrics.json


## 12. Benchmark Results Summary & Hypothesis Outcomes

Comparison of all five pre-race starting grid benchmark models across the **Primary Benchmark Cohort (2018–2024, 149 rounds)** and **Full Series (2010–2026, 346 rounds)**.

### Key Scientific Finding: Proper Scoring Rules vs. Argmax Accuracy
- **Model 3 (Logistic Regression with Grid + Driver/Constructor Form)** achieves the best probabilistic performance (Log Loss: `1.355201`, Brier Score: `0.612698`).
- **Model 4 (LightGBM GBDT)** achieves higher argmax accuracy (Top-1: `53.02%` vs `51.68%`), but performs **worse on proper scoring rules** (Log Loss: `1.409149` vs `1.355201`, Brier: `0.628561` vs `0.612698`).
- **Scientific Implication:** LightGBM exhibits tail probability distortion / slight overconfidence when uncalibrated raw scores are normalized, illustrating the critical necessity of evaluating sports forecasting models via proper scoring rules rather than top-1 accuracy alone.


In [12]:
# Compile comprehensive comparison table across Models 0 to 4
models_summary = [
    {
        "Model": "Model 0: Uniform Baseline ($1 / N_r$)",
        "Family": "Uniform",
        "Primary Log Loss": res_m0_primary["mean_log_loss"],
        "Primary Brier": res_m0_primary["mean_brier_score"],
        "Primary Fract Top-1": res_m0_primary["fractional_top1"],
        "Primary Tie-Aware Top-1": res_m0_primary["tie_aware_top1"],
    },
    {
        "Model": "Model 1: Empirical Grid Frequency",
        "Family": "Empirical Table",
        "Primary Log Loss": res_m1_primary["mean_log_loss"],
        "Primary Brier": res_m1_primary["mean_brier_score"],
        "Primary Fract Top-1": res_m1_primary["fractional_top1"],
        "Primary Tie-Aware Top-1": res_m1_primary["tie_aware_top1"],
    },
    {
        "Model": "Model 2: Grid-Only Logistic Regression",
        "Family": "Linear / Softmax",
        "Primary Log Loss": primary_metrics["mean_log_loss"],
        "Primary Brier": primary_metrics["mean_brier_score"],
        "Primary Fract Top-1": primary_metrics["fractional_top1"],
        "Primary Tie-Aware Top-1": primary_metrics["tie_aware_top1"],
    },
    {
        "Model": "Model 3: Grid + Driver/Constructor Form Logit",
        "Family": "Linear / Softmax",
        "Primary Log Loss": primary_metrics["mean_log_loss"],
        "Primary Brier": primary_metrics["mean_brier_score"],
        "Primary Fract Top-1": primary_metrics["fractional_top1"],
        "Primary Tie-Aware Top-1": primary_metrics["tie_aware_top1"],
    },
    {
        "Model": "Model 4: LightGBM Gradient Boosted Trees",
        "Family": "GBDT / Tree",
        "Primary Log Loss": primary_metrics_m4["mean_log_loss"],
        "Primary Brier": primary_metrics_m4["mean_brier_score"],
        "Primary Fract Top-1": primary_metrics_m4["fractional_top1"],
        "Primary Tie-Aware Top-1": primary_metrics_m4["tie_aware_top1"],
    },
]

# Note: Load verified Model 2 metrics specifically from JSON to ensure exact separation from Model 3 variable names
with open(OUTPUT_DIR / "model2_logistic_metrics.json") as f:
    m2_saved = json.load(f)["primary_cohort_metrics"]
models_summary[2]["Primary Log Loss"] = m2_saved["mean_log_loss"]
models_summary[2]["Primary Brier"] = m2_saved["mean_brier_score"]
models_summary[2]["Primary Fract Top-1"] = m2_saved["fractional_top1"]
models_summary[2]["Primary Tie-Aware Top-1"] = m2_saved["tie_aware_top1"]

with open(OUTPUT_DIR / "model3_metrics.json") as f:
    m3_saved = json.load(f)["primary_cohort_metrics"]
models_summary[3]["Primary Log Loss"] = m3_saved["mean_log_loss"]
models_summary[3]["Primary Brier"] = m3_saved["mean_brier_score"]
models_summary[3]["Primary Fract Top-1"] = m3_saved["fractional_top1"]
models_summary[3]["Primary Tie-Aware Top-1"] = m3_saved["tie_aware_top1"]

df_comparison = pd.DataFrame(models_summary)
print("=== EXP-0002 BENCHMARK COMPARISON TABLE (Primary Cohort: 2018–2024, 149 Races) ===")
display(df_comparison)


=== EXP-0002 BENCHMARK COMPARISON TABLE (Primary Cohort: 2018–2024, 149 Races) ===


,Model,Family,Primary Log Loss,Primary Brier,Primary Fract Top-1,Primary Tie-Aware Top-1
0,Model 0: Uniform Baseline ($1 / N_r$),Uniform,2.995388,0.949982,0.050018,1.000000
1,Model 1: Empirical Grid Frequency,Empirical Table,1.583766,0.682312,0.496644,0.496644
2,Model 2: Grid-Only Logistic Regression,Linear / Softmax,1.594207,0.683382,0.496644,0.496644
3,Model 3: Grid + Driver/Constructor Form Logit,Linear / Softmax,1.355201,0.612698,0.516779,0.516779
4,Model 4: LightGBM Gradient Boosted Trees,GBDT / Tree,1.409149,0.628561,0.530201,0.530201
